# Regression: Tore und Marktwert – mit und ohne Spielerposition

**Lernziel:** Vergleichen Sie drei lineare Regressionsmodelle mit demselben Datensatz:

1. **Modell A – einfache Regression:** Marktwert ~ Tore (eine Linie für alle Spieler).
2. **Modell B – multiple Regression:** Marktwert ~ Tore + Position (parallele Linien, unterschiedliche Ausgangswerte).
3. **Modell C – positionsabhängige Regression:** Marktwert ~ Tore + Position + Tore × Position (unterschiedliche Steigungen je Position).

Das dritte Modell zeigt denselben linearen Zusammenhang wie getrennte einfache Regressionen je Position, sofern alle Positionen berücksichtigt werden. **Keine polynomialen Terme** werden verwendet.

Alle Modelle werden mit **demselben Trainings-/Testsplit** verglichen; die Modellwahl erfolgt hier zu Lehrzwecken anhand des Test-RMSE. Für eine endgültige Leistungsbewertung nach Modellwahl wäre zusätzlich ein unabhängiger Testdatensatz sinnvoll.

## 1. CSV-Datei hochladen
Nutzen Sie z. B. die Spielerdatei mit den Spalten `goalsTotal`, `marketvalue` und `position`.

In [ ]:
# @title CSV hochladen { display-mode: "form" }
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from IPython.display import display

uploaded = files.upload()
if not uploaded:
    raise ValueError("Bitte eine CSV-Datei hochladen.")
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=None, engine='python')
print(f"Datei: {filename} | Zeilen: {len(df):,}")
print('Spalten:', df.columns.tolist())
display(df.head())

## 2. Konfiguration
Diese drei Spaltennamen können für andere Datensätze angepasst werden. Alle weiteren Zellen laufen danach ohne Änderungen.

In [ ]:
# ===== STUDIERENDE: HIER ANPASSEN =====
X_VARIABLE = 'goalsTotal'        # numerisches Feature
Y_VARIABLE = 'marketvalue'       # numerische Zielvariable
POSITION_VARIABLE = 'position'  # kategoriales Feature
TEST_SIZE = 0.25
RANDOM_STATE = 42
# ======================================


## 3. Daten vorbereiten und explorieren
Wir betrachten dieselben Daten zunächst insgesamt und anschließend getrennt nach Position.

In [ ]:
# @title Daten prüfen und visualisieren { display-mode: "form" }
required = [X_VARIABLE, Y_VARIABLE, POSITION_VARIABLE]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Fehlende Spalten: {missing}. Verfügbare Spalten: {df.columns.tolist()}")
data = df[required].copy()
data[X_VARIABLE] = pd.to_numeric(data[X_VARIABLE], errors='coerce')
data[Y_VARIABLE] = pd.to_numeric(data[Y_VARIABLE], errors='coerce')
data = data.replace([np.inf, -np.inf], np.nan).dropna()
data[POSITION_VARIABLE] = data[POSITION_VARIABLE].astype(str).str.strip()
data = data[data[POSITION_VARIABLE] != '']
positions = sorted(data[POSITION_VARIABLE].unique())
if len(positions) < 2:
    raise ValueError('Für den Vergleich benötigen wir mindestens zwei Positionen.')
print(f'Beobachtungen: {len(data):,} | Positionen: {positions}')
display(data.groupby(POSITION_VARIABLE).agg(
    Anzahl=(Y_VARIABLE,'size'),
    Mittlerer_Marktwert=(Y_VARIABLE,'mean'),
    Mittlere_Tore=(X_VARIABLE,'mean')
).round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(data[X_VARIABLE], data[Y_VARIABLE]/1e6, alpha=.15, s=13)
axes[0].set(title='Alle Positionen', xlabel=X_VARIABLE, ylabel=f'{Y_VARIABLE} (Mio.)')
for pos in positions:
    sub = data[data[POSITION_VARIABLE] == pos]
    axes[1].scatter(sub[X_VARIABLE], sub[Y_VARIABLE]/1e6, alpha=.20, s=12, label=pos)
axes[1].set(title='Nach Position getrennt', xlabel=X_VARIABLE, ylabel=f'{Y_VARIABLE} (Mio.)')
axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Modelle trainieren

**Modell A:** Eine gemeinsame Regressionsgerade.  
**Modell B:** Dummy-Variablen für die Position; alle Linien haben dieselbe Steigung.  
**Modell C:** Zusätzlich Interaktionen zwischen Toren und Position; je Position ist eine eigene Steigung möglich.

Die **Referenzposition** ist die alphabetisch erste Position. Im Modell B/C zeigen die Dummy-Koeffizienten Unterschiede zur Referenz.

In [ ]:
# @title Alle drei Modelle trainieren und RMSE vergleichen { display-mode: "form" }
train, test = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)
reference = positions[0]
other_positions = positions[1:]

def make_features(frame, model):
    out = pd.DataFrame(index=frame.index)
    out[X_VARIABLE] = frame[X_VARIABLE].astype(float)
    if model in ('B', 'C'):
        for pos in other_positions:
            dummy = (frame[POSITION_VARIABLE] == pos).astype(float)
            out[f'Position: {pos}'] = dummy
            if model == 'C':
                out[f'{X_VARIABLE} × {pos}'] = out[X_VARIABLE] * dummy
    return out

models = {}
results = []
for key, name in [('A', 'Nur Tore'), ('B', 'Tore + Position'), ('C', 'Tore + Position + Interaktionen')]:
    X_train = make_features(train, key)
    X_test = make_features(test, key)
    reg = LinearRegression()
    reg.fit(X_train, train[Y_VARIABLE])
    pred_train = reg.predict(X_train)
    pred_test = reg.predict(X_test)
    train_rmse = np.sqrt(mean_squared_error(train[Y_VARIABLE], pred_train))
    test_rmse = np.sqrt(mean_squared_error(test[Y_VARIABLE], pred_test))
    models[key] = reg
    results.append({'Modell':key, 'Beschreibung':name, 'Anzahl Koeffizienten (ohne Intercept)': X_train.shape[1],
                    'RMSE Training (Mio.)': train_rmse/1e6,
                    'RMSE Test (Mio.)': test_rmse/1e6})
comparison = pd.DataFrame(results)
print(f'Training: {len(train):,} | Test: {len(test):,} | Referenzposition: {reference}')
display(comparison.round(3))
print('Niedrigster Test-RMSE:', comparison.loc[comparison['RMSE Test (Mio.)'].idxmin(), 'Modell'])

## 5. Regressionsgeraden visuell vergleichen
Die Punkte sind die **Testdaten**; die Linien sind ausschließlich auf den **Trainingsdaten** geschätzt. Die Linien von Modell B verlaufen parallel. Modell C erlaubt unterschiedliche Steigungen.

In [ ]:
# @title Drei Modelle im Vergleich darstellen { display-mode: "form" }
fig, axes = plt.subplots(1, 3, figsize=(18, 4.7), sharey=True)
x_min, x_max = data[X_VARIABLE].min(), data[X_VARIABLE].max()
grid = np.linspace(x_min, x_max, 180)
colors = plt.cm.tab10(np.linspace(0, .85, len(positions)))
for ax, (key, heading) in zip(axes, [('A','A: Nur Tore'),('B','B: Tore + Position'),('C','C: Positionsabhängige Steigung')]):
    for pos, color in zip(positions, colors):
        sub = test[test[POSITION_VARIABLE] == pos]
        ax.scatter(sub[X_VARIABLE], sub[Y_VARIABLE]/1e6, s=11, alpha=.14, color=color)
        new = pd.DataFrame({X_VARIABLE:grid, POSITION_VARIABLE:pos})
        pred = models[key].predict(make_features(new, key))/1e6
        ax.plot(grid, pred, lw=2.3, color=color, label=pos)
    ax.set_title(heading)
    ax.set_xlabel(X_VARIABLE)
    ax.grid(alpha=.15)
axes[0].set_ylabel(f'{Y_VARIABLE} (Mio.)')
axes[-1].legend(loc='best', fontsize=9)
plt.tight_layout(); plt.show()

## 6. Geschätzte Regressionsgleichungen
Hier werden **Achsenabschnitt und Steigung je Position** berechnet. So erkennen Sie direkt, welche Annahmen die drei Modelle treffen.

In [ ]:
# @title Gleichungen je Modell und Position anzeigen { display-mode: "form" }
rows = []
for key in ['A','B','C']:
    for pos in positions:
        sample = pd.DataFrame({X_VARIABLE:[0.0, 1.0], POSITION_VARIABLE:[pos,pos]})
        y0,y1 = models[key].predict(make_features(sample,key))
        rows.append({'Modell':key, 'Position':pos, 'Intercept (Mio.)': y0/1e6,
                     'Steigung (Mio. je Tor)': (y1-y0)/1e6})
equations = pd.DataFrame(rows)
display(equations.round(4))
for key in ['A','B','C']:
    print(f'\nMODELL {key}')
    for row in equations[equations['Modell']==key].itertuples(index=False):
        print(f'{row[1]}: Marktwert (Mio.) = {row[2]:.4f} {row[3]:+.4f} × {X_VARIABLE}')

## 7. Fehlervergleich: Wo gewinnt welches Modell?

In [ ]:
# @title Trainings- und Testfehler visualisieren { display-mode: "form" }
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comparison)); width=.35
ax.bar(x-width/2, comparison['RMSE Training (Mio.)'], width, label='Training')
ax.bar(x+width/2, comparison['RMSE Test (Mio.)'], width, label='Test')
ax.set_xticks(x, ['A: Nur Tore', 'B: + Position', 'C: + Interaktionen'])
ax.set_ylabel('RMSE (Mio.)'); ax.set_title('Vorhersagefehler auf denselben Datensplits')
ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()

## 8. Aufgaben zur Interpretation

1. **Modell A:** Wie verändert sich der vorhergesagte Marktwert bei einem zusätzlichen Tor? Ist dieser Zusammenhang für jede Position gleich?
2. **Modell B:** Wie unterscheiden sich die vorhergesagten Marktwerte für Defender, Midfielder und Forward **bei gleicher Toranzahl**? Warum sind die Linien parallel?
3. **Modell C:** Unterscheidet sich die Steigung je Position? Welche Position zeigt den stärksten geschätzten Zusammenhang zwischen Toren und Marktwert?
4. **Modellvergleich:** Wie verändert sich der Test-RMSE von A zu B und C? Rechtfertigt die Verbesserung die höhere Modellkomplexität?
5. **Reflexion:** Warum ist die Interpretation „ein zusätzliches Tor *verursacht* eine Marktwertsteigerung von X Euro“ nicht zulässig?

**Hinweise:** Dies ist eine beobachtende Analyse, keine Kausalanalyse. Die Regressionen sind linear. Für komplexere Funktionsverläufe (z. B. Polynome) wäre ein erweitertes Modell nötig. Die Ergebnisse hängen außerdem vom Datensatz und Train-/Testsplit ab.